# Stage 3: the development year

**Reads:** `data/gravity_clean.csv`, saved by stage 2. In the full run, also the two files of the last run of part 1, in `results/checkpoint/inputs/` (or `results/checkpoint/rerun/inputs/` if the checkpoint called for a rerun).

**Writes:**
- `results/dev_runs.csv`: one row per configuration and training draw, with its test deviance, the other scores, its training deviance, its seconds, and for `ppml` its coefficients.
- `results/dev_components.csv`: one row per training size and configuration, with the mean test deviance split into variance and remainder.

Both are read by `03b_checkpoint.ipynb` and stage 4.

The stage runs twice (section 6 of `docs/build_plan.md`). First part 1, `python code/run_all.py --part 1`: the training sizes up to `CHECKPOINT_MAX`, before the checkpoint. Then, after the checkpoint, every size, with whatever the checkpoint fixed. At each size, every draw takes one training sample from the training pool, fits every model on it (`ppml`, six forests, one boosted model for each depth, `mean` and `tree`, and from the full run on the ridge line `ridge`, which Yu added at the checkpoint, decision 20) and scores every configuration on the same test set (section 1 of the plan gives the counts, and decision 19 the counts after the checkpoint). Every draw is keyed by year, split, size and draw, so the full run repeats the draws of part 1 exactly, and checks that it does.

*Drafted with Claude (Anthropic), which was asked for stage 3 of the build plan: the test split, the training draws at every size with the 13 fits of each, the rows of `dev_runs.csv`, the split of the error into variance and remainder in `dev_components.csv`, the checks of section 8 that belong to this stage, and the comparison of the full run with part 1. It replaces the placeholder of 18 October 2026. Later asked for (21 October 2026, decision 19, "Apply"; prepared on 20 October before Yu's answer): the text made independent of the size of the grid that the checkpoint extended. Later asked for (21 October 2026, decision 20, "Add it"): the ridge line fitted on every draw, and the penalty it chose printed for each size.*

In [ ]:
from helpers import (CHECKPOINT_MAX, CONFIGURATIONS, DATA, DEV_YEAR, DRAWS, GBM_DEPTHS, GBM_TREE_COUNTS, N_JOBS,
                     PART, PREDICTORS, QUICK, RESULTS, RIDGE_LINE, SIZES, TARGET, check_predictions, component_rows,
                     components_table, draw_rows, draw_sample, largest_relative_difference, make_gbm,
                     projected_seconds, run_draw, run_draws, runs_table, seed_int, split_pairs, test_groups)

import time

import numpy as np
import pandas as pd

# Part 1 runs the sizes up to CHECKPOINT_MAX; without ADA_PART every size runs.
sizes = [n for n in SIZES if PART is None or n <= CHECKPOINT_MAX]
if PART is not None:
    print(f"Part {PART}, the training sizes up to {CHECKPOINT_MAX}:", sizes)
elif QUICK:
    print("Quick mode, every quick-mode training size:", sizes)
else:
    print("Full run, every training size:", sizes)
print(f"{DRAWS} draws per size, {len(CONFIGURATIONS)} configurations per draw, {N_JOBS} worker(s)")

# The base experiment of the development year uses split 0; stage 6 repeats it with split 1.
VARIANT = "base"
SPLIT = 0
RUNS = RESULTS / "dev_runs.csv"
COMPONENTS = RESULTS / "dev_components.csv"

## Step 1. Split the development year into a training pool and a test set

A random quarter of the unordered country pairs goes to the test set, with all its rows (`split_pairs` in `helpers.py`). Both directions of a pair, such as France to Italy and Italy to France, land on the same side, so no model is tested on the mirror image of a row it was trained on. The training samples of every size are drawn from the pool; the test set stays the same for every size, draw and model.

In [ ]:
# Country identifiers are text, as in stage 2: no value is read as missing.
table = pd.read_csv(DATA / "gravity_clean.csv", dtype={"country_id_o": str, "country_id_d": str, "pair_id": str},
                    keep_default_na=False, na_values=[""])
pool, test = split_pairs(table, DEV_YEAR, SPLIT)
print(f"{DEV_YEAR}: {(table['year'] == DEV_YEAR).sum():,} rows")
print(f"  training pool: {len(pool):,} rows, {pool['pair_id'].nunique():,} pairs")
print(f"  test set:      {len(test):,} rows, {test['pair_id'].nunique():,} pairs")

X_pool = pool[PREDICTORS].to_numpy(dtype=float)
y_pool = pool[TARGET].to_numpy(dtype=float)
# The ridge line keeps both directions of a pair in one fold of its cross-validation.
pair_pool = pool["pair_id"].to_numpy()
X_test = test[PREDICTORS].to_numpy(dtype=float)
y_test = test[TARGET].to_numpy(dtype=float)

## Step 2. Check the largest training size

The largest size has to be at most a quarter of the pool (section 6, stage 3 step 2). If it is not, the stage stops here: the size is not shrunk silently.

In [ ]:
largest = max(sizes)
assert largest <= len(pool) / 4, f"The largest size, {largest:,}, is over a quarter of the pool ({len(pool):,} rows). Stop and report."
print(f"Largest size {largest:,}: {largest / len(pool):.1%} of the pool")

## Three groups of test rows

A few very large flows can carry most of the deviance. To see where each model's error comes from, the test rows are cut once into three groups by the size of their flow: the largest 1%, the next 9% and the rest. Each draw records the deviance of each group, summed over its rows and divided by the number of test rows, so the three add up to the test deviance (stage 3 step 5).

In [ ]:
groups = test_groups(y_test)
for group, name in enumerate(["largest 1%", "next 9%", "rest"]):
    in_group = groups == group
    print(f"  {name:<10}: {in_group.sum():>5,} test rows, {y_test[in_group].sum() / y_test.sum():.1%} of the test flow")

## Check: staged prediction of the boosted trees (section 8)

Each boosted model is fitted once with the largest number of trees, and its predictions after fewer trees are read off the same fit (`staged_predict`). On the first draw, the prediction after the smallest number of trees is compared with a model fitted with only that many trees. They must be equal.

In [ ]:
n, draw = sizes[0], 0
positions = draw_sample(len(y_pool), DEV_YEAR, SPLIT, n, draw)
seed = seed_int("model", DEV_YEAR, SPLIT, n, draw)
fewest = GBM_TREE_COUNTS[0]
for depth in GBM_DEPTHS:
    staged = make_gbm(depth, seed).fit(X_pool[positions], y_pool[positions])
    for trees, staged_prediction in enumerate(staged.staged_predict(X_test), start=1):
        if trees == fewest:
            break
    fresh = make_gbm(depth, seed, trees=fewest).fit(X_pool[positions], y_pool[positions])
    assert np.array_equal(staged_prediction, fresh.predict(X_test)), f"Staged prediction differs from a fresh fit at depth {depth}."
print(f"n = {n}, draw {draw}: the staged prediction after {fewest} trees equals a fresh fit with {fewest} trees at every depth")

## Step 3. Time one draw

One draw at the largest size of this run is timed, its fits and predictions together (`run_draw`), and the minutes of this run of the stage (part 1 only, in part 1) are projected from it with the ratios between sizes measured in section 9 of the plan. The notebook does not wait: before the full run in Nuvolos, the projection of the checkpoint (item 8) is read, and Yu is asked if it is over 30 minutes.

In [ ]:
start = time.perf_counter()
run_draw(X_pool, y_pool, pair_pool, X_test, y_test, DEV_YEAR, SPLIT, largest, 0)
timing = time.perf_counter() - start
projected = sum(DRAWS * projected_seconds(timing, largest, n) for n in sizes) / 60
print(f"One draw at n = {largest:,}: {timing:.1f} seconds")
print(f"Projected time of this run of stage 3: {projected:.0f} minutes on one worker"
      + (f", about {projected / N_JOBS:.0f} with {N_JOBS} workers" if N_JOBS > 1 else ""))

## Steps 4 to 7. The draws

For every size and every draw (`run_draw` in `helpers.py`):
- n rows are sampled from the pool without replacement;
- every model is fitted on that one sample, and every configuration predicts the test rows. The ridge line (`RIDGE_LINE`, decision 20) first picks its penalty by 5-fold cross-validation inside the sample, with the folds keeping both directions of a pair together, then is refitted on the whole sample at that penalty (`fit_ridge`, section 6 of the plan, Ridge line);
- every prediction is checked to be positive and finite;
- one row per configuration is kept for `dev_runs.csv` (`draw_rows`). Its seconds are those of the fit alone, as stage 3 step 5 defines them; for `gbm`, those of the one fit at its depth, and for `rf` they include the out-of-bag predictions that scikit-learn computes inside the fit; for `ridge`, those of its 31 fits summed.

The predictions themselves are not saved. For every configuration and test row, the notebook keeps the running sum of the predictions over the draws and the running sum of their logs. After the last draw of a size, these give the split of the mean test deviance into variance and remainder (`split_error`, section 7), one row per configuration for `dev_components.csv`.

After each size, the notebook prints the checks on `ppml` of section 8: how many fits scikit-learn warned had fallen back from Newton-Cholesky to L-BFGS, how many draws had an indicator that was constant in the sample (and so dropped from that fit), and the largest relative difference between the predicted and the observed training total, which unpenalised PPML keeps at the level of rounding errors unless a fit fell back (section 8). With the ridge line, it also prints how many draws chose each penalty.

In [ ]:
runs, components = [], []
for n in sizes:
    start = time.perf_counter()
    sum_predicted = np.zeros((len(CONFIGURATIONS), len(y_test)))
    sum_log_predicted = np.zeros((len(CONFIGURATIONS), len(y_test)))
    size_rows = []
    for draw, result in enumerate(run_draws(X_pool, y_pool, pair_pool, X_test, y_test, DEV_YEAR, SPLIT, n, DRAWS)):
        keys = {"variant": VARIANT, "year": DEV_YEAR, "split": SPLIT, "size": n, "draw": draw}
        check_predictions(result["test_predicted"], keys)
        sum_predicted += result["test_predicted"]
        sum_log_predicted += np.log(result["test_predicted"])
        size_rows.extend(draw_rows(result, y_test, groups, keys))
    size_runs = runs_table(size_rows)
    size_keys = {"variant": VARIANT, "year": DEV_YEAR, "split": SPLIT, "size": n}
    components.extend(component_rows(sum_predicted, sum_log_predicted, y_test, size_runs, n, len(y_pool), size_keys))
    runs.append(size_runs)

    ppml = size_runs[size_runs["config"] == "ppml"]
    print(f"n = {n:>5,}: {DRAWS} draws in {(time.perf_counter() - start) / 60:.1f} minutes. "
          f"ppml: {int(ppml['lbfgs_fallback'].sum())} fit(s) fell back to L-BFGS, "
          f"{int((ppml['constant_columns'] > 0).sum())} draw(s) with a constant indicator, "
          f"largest relative difference of the training total {(ppml['train_total_ratio'] - 1).abs().max():.1e}")
    if RIDGE_LINE:
        chosen = size_runs.loc[size_runs["config"] == "ridge", "ridge_alpha"].value_counts().sort_index()
        print("           ridge: draws that chose each penalty: "
              + ", ".join(f"{alpha:g}: {count}" for alpha, count in chosen.items()))

runs = pd.concat(runs, ignore_index=True)
components = components_table(components)
print(f"dev_runs.csv: {len(runs):,} rows; dev_components.csv: {len(components):,} rows")

## Full run only: compare with part 1

Every draw is keyed by year, split, size and draw, so the full run repeats the draws of part 1. The rows of the part-1 sizes must equal those of the last run of part 1 apart from the seconds, within a relative difference of 1e-9 (section 6, stage 3). The comparison runs over the rows and columns the two files share, before anything is written: if it fails, the stage stops and the files of the last run stay as they were. Part 1 and quick mode compare nothing.

In [ ]:
if PART is None and not QUICK:
    checkpoint = RESULTS / "checkpoint"
    part1 = checkpoint / "rerun" / "inputs"
    if not part1.exists():
        part1 = checkpoint / "inputs"
    assert (part1 / "dev_runs.csv").exists(), f"No files of part 1 in {part1}: run part 1 and the checkpoint first."
    print(f"Part 1 read from {part1.relative_to(RESULTS.parent)}")
    for new, name, keys in [(runs, "dev_runs.csv", ["variant", "year", "split", "size", "draw", "config"]),
                            (components, "dev_components.csv", ["variant", "year", "split", "size", "config"])]:
        # round_trip reads every number back exactly as part 1 wrote it.
        old = pd.read_csv(part1 / name, float_precision="round_trip")
        difference, shared = largest_relative_difference(new, old, keys)
        print(f"  {name}: {shared:,} rows shared out of {len(old):,} in part 1; "
              f"largest relative difference {difference:.1e}")
        assert shared > 0, f"{name}: no row in common with part 1."
        assert difference <= 1e-9, f"{name}: the full run differs from part 1 by {difference:.1e}. Stop and report."
else:
    print("Part 1 or quick mode: nothing to compare.")

## Write the two files

In [ ]:
RESULTS.mkdir(exist_ok=True)
runs.to_csv(RUNS, index=False, lineterminator="\n")
components.to_csv(COMPONENTS, index=False, lineterminator="\n")
print(f"Wrote {RUNS.relative_to(RESULTS.parent)} and {COMPONENTS.relative_to(RESULTS.parent)}")